In [1]:
from utils.metrics import mse_loss, dice_loss, precision_recall_f1, pixel_accuracy, point_matching

In [11]:
# Check with images from Simulation to seed matching, using vae
# Firstly we check the performance on the sole simulation dataset

from utils.config import OUTPUT_DIR_SIMTOSEED_MORESEEDS_3ROTREPS_SAVED, SEED_FOLDER_SEEDTOSIM, SEED_FOLDER_SEEDTOSIM_TEST_2
import os 
import torch
import cv2
import numpy as np
from utils.preprocess import preprocess_seed

prediction_files= sorted(os.listdir(OUTPUT_DIR_SIMTOSEED_MORESEEDS_3ROTREPS_SAVED))
gt_files=  sorted(sorted(os.listdir(SEED_FOLDER_SEEDTOSIM))[50000:50100] + sorted(os.listdir(SEED_FOLDER_SEEDTOSIM_TEST_2)))


mse_loss_list= []
dice_loss_list= []
precision_recall_f1_list= []
pixel_accuracy_list= []

precision_rtol_list= []
accuracy_rtol_list= []
f1_rtol_list= []

# prediction files has output prefix, and gt files has input prefix, apart from that they match

pred_img_tensor_list= []
gt_img_tensor_list= []
for pred_file, gt_file in zip(prediction_files, gt_files):

    # assert pred_file.replace('Output', '') == gt_file.replace('Input', ''), f"File mismatch: {pred_file} vs {gt_file}"

    pred_img= cv2.imread(os.path.join(OUTPUT_DIR_SIMTOSEED_MORESEEDS_3ROTREPS_SAVED, pred_file), cv2.IMREAD_GRAYSCALE)


    # gt files are distributed across two folders, first read from the first folder and if not found read from the second folder
    gt_img_path = os.path.join(SEED_FOLDER_SEEDTOSIM, gt_file)
    if not os.path.exists(gt_img_path):
        gt_img_path = os.path.join(SEED_FOLDER_SEEDTOSIM_TEST_2, gt_file)
    gt_img= preprocess_seed(gt_img_path, left_crop=2, right_crop=3, img_length=32, img_width=32)

    # convert to tensors 
    pred_img_tensor = torch.from_numpy(pred_img)
    gt_img_tensor = torch.from_numpy(gt_img)

    precision, recall, f1_score = point_matching((pred_img_tensor),(gt_img_tensor), tolerance_radius=4, algorithm='hungarian')

    precision_rtol_list.append(precision)
    accuracy_rtol_list.append(recall)
    f1_rtol_list.append(f1_score)
    
    mse_value = mse_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    dice_value = dice_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    precision, recall, f1_score = precision_recall_f1(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    pixel_acc = pixel_accuracy(torch.from_numpy(pred_img), torch.from_numpy(gt_img))

    mse_loss_list.append(mse_value.item())
    dice_loss_list.append(dice_value.item())
    precision_recall_f1_list.append((precision, recall, f1_score))
    pixel_accuracy_list.append(pixel_acc)


    pred_img_tensor= torch.from_numpy(pred_img)
    gt_img_tensor= torch.from_numpy(gt_img)

    pred_img_tensor_list.append(pred_img_tensor)
    gt_img_tensor_list.append(gt_img_tensor)





In [12]:
# For simulations

precision_avg = np.mean([pr[0] for pr in precision_recall_f1_list])
recall_avg = np.mean([pr[1] for pr in precision_recall_f1_list])
f1_avg = np.mean([pr[2] for pr in precision_recall_f1_list])

precision_std= np.std([pr[0] for pr in precision_recall_f1_list])
recall_std= np.std([pr[1] for pr in precision_recall_f1_list])
f1_std= np.std([pr[2] for pr in precision_recall_f1_list])  




print(f"MSE Mean: {np.mean(mse_loss_list)} Std: {np.std(mse_loss_list)}")
print(f"DICE loss Mean: {np.mean(dice_loss_list)} Std :{np.std(dice_loss_list)}")
print(f"Precision : {precision_avg:.3f} ± {precision_std:.3f}")
print(f"Recall : {recall_avg:.3f} ± {recall_std:.3f}")
print(f"F1 Score : {f1_avg:.3f} ± {f1_std:.3f}")
print(f"Pixel Accuracy average: {np.mean(pixel_accuracy_list):.3f} ± {np.std(pixel_accuracy_list):.3f}")

MSE Mean: 0.0009765625 Std: 0.0032539912054787338
DICE loss Mean: 0.021951199134933614 Std :0.05444052272451495
Precision : 0.987 ± 0.035
Recall : 0.971 ± 0.074
F1 Score : 0.978 ± 0.054
Pixel Accuracy average: 0.999 ± 0.003


In [13]:
# Simulations, tolerance point matching results (radius=3, Hungarian)

print(f"Tolerance Precision {np.mean(precision_rtol_list)} ± {np.std(precision_rtol_list)}")
print(f"Tolerance Recall {np.mean(accuracy_rtol_list)} ± {np.std(accuracy_rtol_list)}")
print(f"Tolerance F1 {np.mean(f1_rtol_list)} ± {np.std(f1_rtol_list)}")

Tolerance Precision 0.9939848530494361 ± 0.021911907829047974
Tolerance Recall 0.9769322141828886 ± 0.05955491123030383
Tolerance F1 0.9842804978283335 ± 0.036967509719381064


In [14]:
# Now on experimental test set  

from utils.config import OUTPUT_DIR_SIMTOSEED_MORESEEDS_3ROTREPS_EXPDATASET_SAVED,SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32 
import os 
import torch
import cv2
import numpy as np
from utils.preprocess import preprocess_seed
import matplotlib.pyplot as plt

prediction_files= sorted(os.listdir(OUTPUT_DIR_SIMTOSEED_MORESEEDS_3ROTREPS_EXPDATASET_SAVED))
gt_files=  sorted(os.listdir(SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32))


mse_loss_list= []
dice_loss_list= []
precision_recall_f1_list= []
pixel_accuracy_list= []

precision_rtol_list= []
accuracy_rtol_list= []
f1_rtol_list= []

# prediction files has output prefix, and gt files has input prefix, apart from that they match

pred_img_tensor_list= []
gt_img_tensor_list= []
for pred_file, gt_file in zip(prediction_files, gt_files):

    # assert pred_file.replace('Output', '') == gt_file.replace('Input', ''), f"File mismatch: {pred_file} vs {gt_file}"

    pred_img= cv2.imread(os.path.join(OUTPUT_DIR_SIMTOSEED_MORESEEDS_3ROTREPS_EXPDATASET_SAVED, pred_file), cv2.IMREAD_GRAYSCALE)
    gt_img= preprocess_seed(os.path.join(SEED_FOLDER_EXPTOSIM_TEST_FIXED_NONFIXED_32X32, gt_file), img_length=32, img_width=32)

    # convert to tensors 
    pred_img_tensor = torch.from_numpy(pred_img)
    gt_img_tensor = torch.from_numpy(gt_img)

    precision, recall, f1_score = point_matching((pred_img_tensor),(gt_img_tensor), tolerance_radius=4, algorithm='hungarian')

    precision_rtol_list.append(precision)
    accuracy_rtol_list.append(recall)
    f1_rtol_list.append(f1_score)
    
    mse_value = mse_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    dice_value = dice_loss(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    precision, recall, f1_score = precision_recall_f1(torch.from_numpy(pred_img), torch.from_numpy(gt_img))
    pixel_acc = pixel_accuracy(torch.from_numpy(pred_img), torch.from_numpy(gt_img))

    mse_loss_list.append(mse_value.item())
    dice_loss_list.append(dice_value.item())
    precision_recall_f1_list.append((precision, recall, f1_score))
    pixel_accuracy_list.append(pixel_acc)

    pred_img_tensor= torch.from_numpy(pred_img)
    gt_img_tensor= torch.from_numpy(gt_img)

    pred_img_tensor_list.append(pred_img_tensor)
    gt_img_tensor_list.append(gt_img_tensor)





In [ ]:
# Experiments 

precision_avg = np.mean([pr[0] for pr in precision_recall_f1_list])
recall_avg = np.mean([pr[1] for pr in precision_recall_f1_list])
f1_avg = np.mean([pr[2] for pr in precision_recall_f1_list])

precision_std= np.std([pr[0] for pr in precision_recall_f1_list])
recall_std= np.std([pr[1] for pr in precision_recall_f1_list])
f1_std= np.std([pr[2] for pr in precision_recall_f1_list])  




print(f"MSE Mean: {np.mean(mse_loss_list)} Std: {np.std(mse_loss_list)}")
print(f"DICE loss Mean: {np.mean(dice_loss_list)} Std :{np.std(dice_loss_list)}")
print(f"Precision : {precision_avg:.3f} ± {precision_std:.3f}")
print(f"Recall : {recall_avg:.3f} ± {recall_std:.3f}")
print(f"F1 Score : {f1_avg:.3f} ± {f1_std:.3f}")
print(f"Pixel Accuracy average: {np.mean(pixel_accuracy_list):.3f} ± {np.std(pixel_accuracy_list):.3f}")



MSE Mean: 0.01327237215909091 Std: 0.009168815580640883
DICE loss Mean: 0.9953597314430006 Std :0.02875176796260965
Precision : 0.006 ± 0.039
Recall : 0.004 ± 0.026
F1 Score : 0.005 ± 0.029
Pixel Accuracy average: 0.987 ± 0.009


In [ ]:
# Experiments, tolerance point matching results (radius=4, Hungarian)

print(f"Tolerance Precision {np.mean(precision_rtol_list)} ± {np.std(precision_rtol_list)}")
print(f"Tolerance Recall {np.mean(accuracy_rtol_list)} ± {np.std(accuracy_rtol_list)}")
print(f"Tolerance F1 {np.mean(f1_rtol_list)} ± {np.std(f1_rtol_list)}")

Tolerance Precision 0.3767885700419701 ± 0.36474908592465405
Tolerance Recall 0.1904890928651343 ± 0.23517051043329412
Tolerance F1 0.2179402791091799 ± 0.2244751634904258
